# Reproduction demo

Runs the central method end to end on a small committed subset of the data: build a BM25 index → search → build the prompts → score the answer options with Gemma 3 270M (without and with the retrieved chunks) and Borealis 270M (without). It uses the same code as the full experiment (`src/norrag/`), and only the files in `demo/data/`, so none of the scripts 01–08 need to be run first.

**Needs:** `pip install -e .` in the project folder, and a Hugging Face token in `.env` (`HF_TOKEN=...`) from an account that has accepted the Gemma licence on huggingface.co/google/gemma-3-270m-it. The first run downloads the two 270M models (about 0.5 GB each).

**Note:** the 270M models are far weaker than the 12B models in the report, and the demo questions are too few for firm conclusions (the report uses all 3,516 questions and a McNemar test). The point is to show that the method runs and what each step does; the report's results are shown at the end for reference.

In [ ]:
import tempfile
import textwrap
import time

import pandas as pd
from tqdm import tqdm

from norrag.config import get_path, load_config
from norrag.metrics import chance_accuracy, letter_mass
from norrag.models import load_model, pick_device
from norrag.pipeline import run_condition
from norrag.retrieval.base import get_retriever
from norrag.retrieval.query import build_query

start = time.time()
config = load_config()
settings = config["demo"]
k = config["qa"]["k"]  # chunks per question in the prompt
demo_data = get_path(config, "demo_data")
questions = pd.read_parquet(demo_data / "questions.parquet")
chunks = pd.read_parquet(demo_data / "chunks.parquet")
results_12b = pd.read_parquet(demo_data / "results_12b.parquet")  # the report's answers to the same questions
print(f"{len(questions)} questions, {len(chunks):,} chunks; scoring on {pick_device()}")

## 1 · Search

Build a BM25 index over the demo chunks and find the top 5 for each question. The chunks are the ones BM25 found for these questions on the full Wikipedia collection (2.5 million chunks) plus 5,000 random others, so the search can find mostly the same chunks as in the full run, though not all: how rare a word is is now counted over far fewer chunks.

In [ ]:
retriever = get_retriever(settings["retriever"], config)
with tempfile.TemporaryDirectory() as folder:  # build() saves the index; here it is only needed in memory
    retriever.build(chunks, folder)
queries = [build_query(row.question, row.options, config["retrieval"]["query_mode"]) for row in questions.itertuples()]
run = {qid: chunk_ids for qid, (chunk_ids, scores) in zip(questions.qid, retriever.search(queries, k=k))}
chunk_texts = chunks.set_index("chunk_id")["text"]

# How many of these top 5 chunks the full run also had in its top 5 (saved with the 12B answers)
full_run = results_12b[results_12b.condition == f"rag_{settings['retriever']}"].set_index("qid").chunk_ids
same = sum(len(set(run[qid]) & set(full_run[qid])) for qid in questions.qid)
print(f"{same} of {k * len(questions)} chunks in the top {k} are the same as in the full run")

## 2 · Score the answer options

For each question and condition, `run_condition` builds the prompt (with the top 5 chunks for retrieval), gives it to the model and reads the probability of each option letter as the first token of the answer. The letter with the highest probability is the model's answer. This is the same function the full experiment uses (`scripts/07_run_qa.py`).

In [ ]:
plan = {"gemma": ["no_rag", f"rag_{settings['retriever']}"], "borealis": ["no_rag"]}  # model: conditions
records = []
for name, conditions in plan.items():
    spec = settings["models"][name]
    model, tokenizer = load_model(spec["id"], spec["revision"])  # downloads the model the first time
    for condition in conditions:
        scored = run_condition(questions, model, tokenizer, name, condition, config, run=run, chunk_texts=chunk_texts)
        records += list(tqdm(scored, total=len(questions), desc=f"{spec['id']} {condition}"))
    del model  # free the memory before the next model
results = pd.DataFrame(records)

## 3 · Results

Correct answers for each condition, from the 270M models in this demo and from the 12B models in the report on the same questions. Letter mass is the share of the model's probability that lands on the option letters (near 100 % means it wants to answer with a letter). Chance is what guessing would give, from the number of options per question.

In [ ]:
names = {("borealis", "no_rag"): "Borealis, no retrieval", ("gemma", "no_rag"): "Gemma, no retrieval",
         ("gemma", f"rag_{settings['retriever']}"): f"Gemma + {settings['retriever'].upper()}"}

def summary(results, label):
    # Correct answers and mean letter mass per condition; "gemma-12b" and "gemma" both count as Gemma
    family = results.model.str.split("-").str[0]
    grouped = results.assign(mass=results.logprobs.apply(letter_mass)).groupby([family, results.condition])
    table = grouped.agg(correct=("correct", "sum"), mass=("mass", "mean"))
    table.index = [names[key] for key in table.index]
    return table.rename(columns={"correct": f"{label}: correct of {len(questions)}", "mass": f"{label}: letter mass"})

comparison = summary(results, "270M (this demo)").join(summary(results_12b, "12B (report)"))
comparison = comparison.loc[list(names.values())]
print(f"Chance: {chance_accuracy(questions.options.str.len()) * len(questions):.1f} of {len(questions)}")
comparison.style.format({column: "{:.0%}" for column in comparison.columns if "mass" in column})

## 4 · One question up close

The prompt Gemma got with the retrieved chunks, and what each model answered.

In [ ]:
qid = questions.qid.iloc[0]
question = questions.set_index("qid").loc[qid]
rag = results[(results.qid == qid) & (results.condition != "no_rag")].iloc[0]
print(rag.prompt)
print("\nCorrect answer:", question.answer)
for frame, size in [(results, "270M"), (results_12b, "12B")]:
    for row in frame[frame.qid == qid].itertuples():
        family = row.model.split("-")[0]
        print(f"  {size:5} {names[(family, row.condition)]:24} chose {row.predicted} ({'right' if row.correct else 'wrong'})")

## The report's results

Accuracy on all 3,516 questions with the 12B models, from `results/tables/accuracy.csv` (made by `notebooks/00_main.ipynb`).

In [ ]:
print(f"The demo took {(time.time() - start) / 60:.1f} minutes")
pd.read_csv(get_path(config, "tables") / "accuracy.csv", dtype={"correct": "Int64"})